# Teste manual — Gold (`GoldOrchestrator`)

Pré-requisito: silver populada (`python run_silver.py daily` ou `one feriados`).

Fluxo: **silver** → `read_silver` / `materialize` → valor gold (SQL-ready ou builder).

O pipeline gold (`run_gold.py`, futuro) decide quando persistir no SQLite.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if (ROOT / "src" / "config").is_dir():
    PROJECT_ROOT = ROOT
elif (ROOT.parent / "src" / "config").is_dir():
    PROJECT_ROOT = ROOT.parent
else:
    raise RuntimeError("Abra o notebook na raiz do repo ou em notebooks/")

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from config import get_settings
from pipelines.gold import BUILDER_NAMES, GoldOrchestrator

settings = get_settings()
orch = GoldOrchestrator()
START, END = "2026-01-01", "2026-01-31"

print("SILVER:", settings.silver_root)
print("Builders:", BUILDER_NAMES)

SILVER: \\BRZ1WS2478\Brazil_Dept\Mesa_Papel\Chila\projetos\brazil_fixed_income_analytics\data\silver
Builders: ('feriados', 'cdi', 'ptax', 'bmf', 'mercado_secundario', 'liquidacoes_mercado', 'leiloes', 'ipca_dict', 'vna_lft')


## 1. Feriados (materializer — SQL-ready)

Silver: `feriados/snapshot=1` (leitura integral, sem datas). Saída: `list[str]` ISO para tabela `FERIADOS`.

In [2]:
silver_feriados = orch.read_feriados()
display(silver_feriados["feriados"].head())

result_feriados = orch.materialize_feriados()
datas = result_feriados.value
print("total feriados:", len(datas))
print("primeiros:", datas[:5])
print("ultimos:", datas[-5:])

,data
0,2001-01-01
1,2001-02-26
2,2001-02-27
3,2001-04-13
4,2001-04-21


total feriados: 1263
primeiros: ['2001-01-01', '2001-02-26', '2001-02-27', '2001-04-13', '2001-04-21']
ultimos: ['2099-10-12', '2099-11-02', '2099-11-15', '2099-11-20', '2099-12-25']


In [3]:
silver_feriados

{'feriados':             data
 0     2001-01-01
 1     2001-02-26
 2     2001-02-27
 3     2001-04-13
 4     2001-04-21
 ...          ...
 1258  2099-10-12
 1259  2099-11-02
 1260  2099-11-15
 1261  2099-11-20
 1262  2099-12-25
 
 [1263 rows x 1 columns]}

## 2. CDI (materializer — SQL-ready)

Silver: partições `cdi/data=YYYY-MM-DD`. Passe a lista de datas (o pipeline define quais carregar).

In [2]:
dates = ["2026-01-15", "2026-01-16"]  # exemplo; em produção o pipeline passa a lista
result_cdi = orch.materialize_cdi(dates)
display(result_cdi.value)
print("linhas:", len(result_cdi.value))

,data_referencia,cdi
0,2026-01-15,14.9
1,2026-01-16,14.9


linhas: 2


## 3. PTAX USD (materializer — SQL-ready)

Silver: partições `ptax/data=YYYY-MM-DD`. Passe a lista de datas (o pipeline define quais carregar).

In [2]:
dates_ptax = ["2026-01-15", "2026-01-16"]  # exemplo; em produção o pipeline passa a lista
result_ptax = orch.materialize_ptax(dates_ptax)
display(result_ptax.value)
print("linhas:", len(result_ptax.value))

,data_referencia,ptax_compra,ptax_venda
0,2026-01-15,5.384,5.3846
1,2026-01-16,5.3792,5.3798


linhas: 2


## 4. BMF ajustes (materializer — SQL-ready)

Silver: partições `ajustes_bmf/data=YYYY-MM-DD`. Passe a lista de datas.

In [2]:
dates_bmf = ["2026-01-15", "2026-01-16"]  # exemplo; em produção o pipeline passa a lista
result_bmf = orch.materialize_bmf(dates_bmf)
display(result_bmf.value)
print("linhas:", len(result_bmf.value))

,data_referencia,data_vencimento,ticker,taxa_ajuste,quantidade_ajuste,codigo_isin
0,2026-01-15,2026-01-15,DAPF26,0.000,100000.00,BRBMEFDAP4M4
1,2026-01-15,2026-02-02,DI1G26,14.894,99341.04,BRBMEFD1I8F3
2,2026-01-15,2026-02-18,DAPG26,9.594,99203.40,BRBMEFDAP553
3,2026-01-15,2026-03-02,DI1H26,14.869,98363.28,BRBMEFD1I8G1
4,2026-01-15,2026-03-16,DAPH26,7.786,98816.93,BRBMEFDAP579
...,...,...,...,...,...,...
124,2026-01-16,2041-01-02,DI1F41,13.677,14881.38,BRBMEFD1I8T4
125,2026-01-16,2045-05-15,DAPK45,7.275,25963.05,BRBMEFDAP3B9
126,2026-01-16,2050-08-15,DAPQ50,7.210,18265.64,BRBMEFDAP3C7
127,2026-01-16,2055-05-17,DAPK55,7.155,13349.68,BRBMEFDAP3D5


linhas: 129


## 5. Mercado secundário (materializer — SQL-ready)

Silver: partições `mercado_secundario/data=YYYY-MM-DD`. Passe a lista de datas.

In [2]:
dates_ms = ["2026-01-15", "2026-01-16"]
result_ms = orch.materialize_mercado_secundario(dates_ms)
display(result_ms.value)
print("linhas:", len(result_ms.value))

,tipo_titulo,data_vencimento,data_referencia,taxa_anbima,intervalo_min_d0,intervalo_max_d0,intervalo_min_d1,intervalo_max_d1,pu,expressao,data_base,codigo_selic,codigo_isin,taxa_compra,taxa_venda,desvio_padrao,status
0,LFT,2026-03-01,2026-01-15,0.0221,-0.0474,0.0670,-0.0479,0.0673,18185.217583,Rentabilidade (% a.a.)/252,2000-07-01,210100,BRSTNCLF1RE0,0.0257,0.0203,0.000320,ATIVO
1,LTN,2026-04-01,2026-01-15,14.7330,14.6900,14.9439,14.6872,14.9385,972.038253,Taxa (% a.a.)/252,2024-01-05,100000,BRSTNCLTN8B5,14.7391,14.7253,0.000000,ATIVO
2,LTN,2026-07-01,2026-01-15,14.4124,14.2789,14.6536,14.2876,14.6586,941.412415,Taxa (% a.a.)/252,2023-01-06,100000,BRSTNCLTN848,14.4177,14.4070,0.002405,ATIVO
3,NTN-B,2026-08-15,2026-01-15,10.2999,9.8912,10.7313,9.9108,10.7557,4594.453474,Taxa (% a.a.)/252,2000-07-15,760199,BRSTNCNTB4U6,10.3119,10.2862,0.001841,ATIVO
4,LFT,2026-09-01,2026-01-15,-0.0147,-0.0399,0.0556,-0.0400,0.0563,18187.363497,Rentabilidade (% a.a.)/252,2000-07-01,210100,BRSTNCLF1RF7,-0.0100,-0.0184,0.000226,ATIVO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,NTN-B,2040-08-15,2026-01-16,7.5204,7.3332,7.6765,7.3610,7.7045,4121.044008,Taxa (% a.a.)/252,2000-07-15,760199,BRSTNCNTB3C6,7.5362,7.5085,0.001048,ATIVO
100,NTN-B,2045-05-15,2026-01-16,7.3854,7.2345,7.5641,7.2378,7.5676,4019.259772,Taxa (% a.a.)/252,2000-07-15,760199,BRSTNCNTB0A6,7.4018,7.3705,0.005640,ATIVO
101,NTN-B,2050-08-15,2026-01-16,7.3229,7.1814,7.5035,7.1820,7.5044,4049.251613,Taxa (% a.a.)/252,2000-07-15,760199,BRSTNCNTB3D4,7.3436,7.3084,0.003933,ATIVO
102,NTN-B,2055-05-15,2026-01-16,7.2700,7.1331,7.4537,7.1304,7.4514,3967.205126,Taxa (% a.a.)/252,2000-07-15,760199,BRSTNCNTB4Q4,7.2865,7.2535,0.000000,ATIVO


linhas: 104


data_referencia        str
cdi                float64
dtype: object

In [3]:
dates_liq = ["2026-01-15", "2026-01-16"]
result_liq = orch.materialize_liquidacoes_mercado(dates_liq)
display(result_liq.value)
print("linhas:", len(result_liq.value))

,tipo_titulo,data_vencimento,data_referencia,qtd_operacoes,qtd_titulos,pu_medio,status
0,LFT,2026-03-01,2026-01-15,29,255257,18185.235146,ATIVO
1,LTN,2026-04-01,2026-01-15,15,1080164,972.043333,ATIVO
2,LTN,2026-07-01,2026-01-15,16,940528,941.430919,ATIVO
3,NTN-B,2026-08-15,2026-01-15,74,479630,4595.095904,ATIVO
4,LFT,2026-09-01,2026-01-15,16,60132,18187.564957,ATIVO
...,...,...,...,...,...,...,...
92,NTN-B,2040-08-15,2026-01-16,1,1469,1672.063464,ATIVO
93,NTN-B,2045-05-15,2026-01-16,121,182658,4015.223357,ATIVO
94,NTN-B,2050-08-15,2026-01-16,2,1865,886.786272,ATIVO
95,NTN-B,2055-05-15,2026-01-16,1,134,665.222696,ATIVO


linhas: 97


In [7]:
from pipelines.gold import GoldOrchestrator

orch = GoldOrchestrator()
result = orch.materialize_leiloes(["2026-05-18", "2026-05-13"])
df = result.value  # DataFrame pronto para INSERT em LEILOES

In [8]:
df

,numero_edital,tipo_titulo,data_vencimento,data_referencia,oferta,quantidade_aceita,percentual_corte,oferta_segunda_volta,financeiro_aceito,financeiro_aceito_segunda_volta,quantidade_aceita_segunda_volta,pu_medio,taxa_media


## 6. IPCA dict (protótipo do builder `ipca_dict`)

Silver:
- `ipca_indice` → índice e variação mensal fechada (SIDRA)
- `projecoes` → `variacao_projetada` ANBIMA (IPCA), com revisões por `data_coleta` / `data_validade`
- `feriados` → ajuste de dia 15 e dias úteis

Lógica espelhada do builder legado (`inicio_fim_mes_ipca`, `dicionario_ipca`): `IPCA_USADO` é fechado derivado do índice quando o último mês fechado é M−1 e a data está antes do dia 15; caso contrário usa projeção do mês IPCA em aberto (`fim_mes_ipca`).

In [17]:
from pipelines.silver.reader import read_partition, list_partition_values

In [74]:
import pandas as pd


def _month_start(as_of: str | pd.Timestamp) -> str:
    ts = pd.Timestamp(as_of).normalize()
    return f"{ts.year:04d}-{ts.month:02d}-01"


def _ultimas_particoes_nao_vazias(
    dataset: str,
    as_of: str | pd.Timestamp,
    n: int = 2,
) -> pd.DataFrame:
    """Últimas ``n`` partições silver não vazias com reference_month <= mês de ``as_of``."""
    limite = _month_start(as_of)
    candidatos = sorted(
        (p for p in list_partition_values(dataset) if p <= limite),
        reverse=True,
    )
    frames: list[pd.DataFrame] = []
    for part in candidatos:
        if len(frames) >= n:
            break
        try:
            df = read_partition(dataset, part)
        except FileNotFoundError:
            continue
        if df is None or df.empty:
            continue
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def silver_projecoes_e_ipca_ultimos_dois(
    as_of: str | pd.Timestamp,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Retorna (projecoes, ipca_indice): cada um é a união das 2 últimas
    partições mensais não vazias no silver até ``as_of``.
    """
    df_proj = _ultimas_particoes_nao_vazias("projecoes", as_of, n=2)
    df_ipca = _ultimas_particoes_nao_vazias("ipca_indice", as_of, n=2)
    return df_proj, df_ipca

In [73]:
import pandas as pd


def projecao_mais_recente(
    df_proj: pd.DataFrame,
    as_of: str | pd.Timestamp,
    *,
    indice: str | None = "IPCA",
    tipo_projecao: str | None = "PROJEÇÕES PARA O MÊS CORRENTE",
    ref_month: str | pd.Timestamp | None = None,
    respeitar_validade: bool = False,
) -> pd.Series:
    """
    Projeção com data_coleta <= as_of mais próxima de as_of (última coleta válida).

    Filtros opcionais: indice, tipo_projecao, ref_month (YYYY-MM-01).
    """
    as_of = pd.Timestamp(as_of).normalize()
    sub = df_proj.copy()

    if indice is not None:
        sub = sub[sub["indice"].astype(str).str.upper() == indice.upper()]
    if tipo_projecao is not None:
        sub = sub[sub["tipo_projecao"] == tipo_projecao]
    if ref_month is not None:
        mes = pd.Timestamp(ref_month).strftime("%Y-%m-%d")[:7] + "-01"
        sub = sub[sub["ref_month"].astype(str).str[:10] == mes]

    sub["data_coleta"] = pd.to_datetime(sub["data_coleta"], errors="coerce").dt.normalize()
    sub = sub[sub["data_coleta"].notna() & (sub["data_coleta"] <= as_of)]

    if respeitar_validade and "data_validade" in sub.columns:
        val = pd.to_datetime(sub["data_validade"], errors="coerce").dt.normalize()
        sub = sub[val.isna() | (val >= as_of)]

    if sub.empty:
        raise ValueError(f"Nenhuma projeção com data_coleta <= {as_of.date()}")

    # Mais próxima de as_of = maior data_coleta entre as <= as_of
    return sub.loc[sub["data_coleta"].idxmax()]


def projecao_mais_recente_valor(
    df_proj: pd.DataFrame,
    as_of: str | pd.Timestamp,
    **kwargs,
) -> tuple[float, pd.Timestamp]:
    row = projecao_mais_recente(df_proj, as_of, **kwargs)
    return (
        float(row["variacao_projetada"]),
        pd.Timestamp(row["data_coleta"]).normalize(),
    )
'''
def projecao_mais_recente_float(df_proj: pd.DataFrame, as_of: str | pd.Timestamp, **kwargs) -> float:
    return float(projecao_mais_recente(df_proj, as_of, **kwargs)["variacao_projetada"])
'''

'\ndef projecao_mais_recente_float(df_proj: pd.DataFrame, as_of: str | pd.Timestamp, **kwargs) -> float:\n    return float(projecao_mais_recente(df_proj, as_of, **kwargs)["variacao_projetada"])\n'

In [75]:
import pandas as pd
from pipelines.gold._io import read_silver_range
from pipelines.gold.orchestrator import GoldOrchestrator

# Constante do legado (NTN-B); futuramente pode vir de série histórica
INDICE_IPCA_DATA_BASE = 1614.62
'''
_TIPO_PROJ_CORRENTE = "PROJEÇÕES PARA O MÊS CORRENTE"
_TIPO_PROJ_POSTERIOR = "PROJEÇÕES PARA O MÊS POSTERIOR"
'''

def _feriados_set() -> set[str]:
    return set(GoldOrchestrator().materialize_feriados().value)


def e_dia_util(data: pd.Timestamp | str, feriados: set[str]) -> bool:
    ts = pd.Timestamp(data).normalize()
    if ts.weekday() >= 5:
        return False
    return ts.strftime("%Y-%m-%d") not in feriados


def adicionar_dias_uteis(
    data: pd.Timestamp | str,
    n_dias: int,
    feriados: set[str],
) -> pd.Timestamp:
    cur = pd.Timestamp(data).normalize()
    step = 1 if n_dias >= 0 else -1
    remaining = abs(int(n_dias))
    while remaining > 0:
        cur += pd.Timedelta(days=step)
        if e_dia_util(cur, feriados):
            remaining -= 1
    return cur.normalize()


def inicio_fim_mes_ipca(
    data: pd.Timestamp | str,
    feriados: set[str] | None = None,
) -> tuple[pd.Timestamp, pd.Timestamp]:
    feriados = feriados if feriados is not None else _feriados_set()
    data = pd.Timestamp(data).normalize()
    dia_15 = {
        "ant": (data - pd.DateOffset(months=1)).replace(day=15).normalize(),
        "atu": data.replace(day=15).normalize(),
        "prox": (data + pd.DateOffset(months=1)).replace(day=15).normalize(),
    }
    for key in dia_15:
        if not e_dia_util(dia_15[key], feriados):
            dia_15[key] = adicionar_dias_uteis(dia_15[key], 1, feriados)
    if data < dia_15["atu"]:
        return dia_15["ant"], dia_15["atu"]
    return dia_15["atu"], dia_15["prox"]


def _pick_ipca_month(
    monthly: pd.DataFrame,
    year: int,
    month: int,
    *,
    strict: bool = True,
) -> pd.Series:
    """Mês exato ou, se ``strict=False``, último ``ref_month`` publicado <= alvo."""
    alvo = pd.Timestamp(year=year, month=month, day=1)
    mask = (monthly["ref_month"].dt.year == year) & (monthly["ref_month"].dt.month == month)
    rows = monthly.loc[mask]
    if not rows.empty:
        return rows.iloc[-1]
    if strict:
        raise KeyError(f"ipca_indice sem ref_month {year:04d}-{month:02d}")
    prior = monthly[monthly["ref_month"] <= alvo]
    if prior.empty:
        raise KeyError(f"ipca_indice sem histórico em ou antes de {year:04d}-{month:02d}")
    return prior.iloc[-1]


def ipca_fechado_from_monthly(
    ipca_monthly: pd.DataFrame,
    as_of: pd.Timestamp | str,
) -> dict[str, float | int]:
    """Último mês fechado = M-1 da data; anterior = M-2 (equivalente ao iloc do SIDRA legado)."""
    as_of = pd.Timestamp(as_of).normalize()
    mes_atual = as_of - pd.DateOffset(months=1)
    mes_anterior = as_of - pd.DateOffset(months=2)

    monthly = ipca_monthly.copy()
    monthly["ref_month"] = pd.to_datetime(monthly["ref_month"])
    monthly = monthly.sort_values("ref_month").reset_index(drop=True)

    mes_limite = mes_atual.to_period("M").to_timestamp()
    cand = monthly[monthly["ref_month"] <= mes_limite]
    if cand.empty:
        cand = monthly[monthly["ref_month"] <= as_of.replace(day=1)]
    if cand.empty:
        cand = monthly
    row_atual = cand.iloc[-1]
    idx_atual = int(monthly.index[monthly["ref_month"] == row_atual["ref_month"]][-1])
    if idx_atual == 0:
        # Histórico curto no lake: ainda monta o dict; ``dicionario_ipca`` usa projeção.
        row_ant = row_atual
    else:
        row_ant = monthly.iloc[idx_atual - 1]

    return {
        "ULTIMO_MES_IPCA": int(row_atual["ref_month"].month),
        "INDICE_IPCA_FECHADO_ATUAL": float(row_atual["ipca_index"]),
        "INDICE_IPCA_FECHADO_ANTERIOR": float(row_ant["ipca_index"]),
        "VAR_IPCA_ATUAL": float(row_atual["ipca_mom"]),
        "VAR_IPCA_ANTERIOR": float(row_ant["ipca_mom"]),
    }


def _projecoes_ipca_validas(
    projecoes: pd.DataFrame,
    as_of: pd.Timestamp,
) -> pd.DataFrame:
    sub = projecoes.loc[projecoes["indice"].astype(str).str.upper() == "IPCA"].copy()
    if sub.empty:
        raise ValueError("projecoes sem linhas IPCA")

    sub["ref_month"] = pd.to_datetime(sub["ref_month"], errors="coerce").dt.normalize()
    sub["data_coleta"] = pd.to_datetime(sub["data_coleta"], errors="coerce").dt.normalize()
    sub["data_validade"] = pd.to_datetime(sub["data_validade"], errors="coerce").dt.normalize()

    sub = sub[sub["ref_month"].notna() & sub["data_coleta"].notna()]
    sub = sub[sub["data_coleta"] <= as_of]
    sub = sub[sub["data_validade"].isna() | (sub["data_validade"] >= as_of)]
    if sub.empty:
        raise ValueError(f"projecoes sem IPCA válido em {as_of.date()}")
    return sub


def _pick_projecao_row(cand: pd.DataFrame) -> pd.Series:
    latest_coleta = cand["data_coleta"].max()
    sub = cand[cand["data_coleta"] == latest_coleta]
    for tipo in (_TIPO_PROJ_CORRENTE, _TIPO_PROJ_POSTERIOR):
        prefer = sub[sub["tipo_projecao"] == tipo]
        if not prefer.empty:
            return prefer.iloc[0]
    return sub.iloc[0]

'''
def ipca_proj_float_from_projecoes(
    projecoes: pd.DataFrame,
    as_of: pd.Timestamp | str,
    fim_mes_ipca: pd.Timestamp,
) -> tuple[float, str]:
    """
    Substitui ``scrap_proj_ipca``.

    1) Tenta ``ref_month`` = mês de ``fim_mes_ipca`` (mês IPCA em aberto).
    2) Se a ANBIMA ainda não publicou esse mês, usa a projeção válida mais recente
       com ``ref_month`` <= mês alvo (ex.: em 18/05 usa 05/2026 até sair 06/2026).

    Retorna (variacao_projetada, ref_month_usado ISO).
    """
    as_of = pd.Timestamp(as_of).normalize()
    mes_alvo = pd.Timestamp(fim_mes_ipca).normalize().replace(day=1)

    sub = _projecoes_ipca_validas(projecoes, as_of)
    cand = sub[sub["ref_month"] == mes_alvo]
    if cand.empty:
        cand = sub[sub["ref_month"] <= mes_alvo]
    if cand.empty:
        cand = sub
    if cand.empty:
        raise ValueError(f"projecoes sem IPCA utilizável para {mes_alvo.date()}")

    ref_usado = cand["ref_month"].max()
    cand = cand[cand["ref_month"] == ref_usado]
    row = _pick_projecao_row(cand)
    return float(row["variacao_projetada"]), ref_usado.strftime("%Y-%m-%d")

'''
def dicionario_ipca(
    as_of: pd.Timestamp | str,
    fechado: dict[str, float | int],
    ipca_proj_float: float,
    feriados: set[str] | None = None,
) -> dict[str, float | int]:
    feriados = feriados if feriados is not None else _feriados_set()
    data = pd.Timestamp(as_of).normalize()

    dia_15_mes_atu = data.replace(day=15).normalize()
    if not e_dia_util(dia_15_mes_atu, feriados):
        dia_15_mes_atu = adicionar_dias_uteis(dia_15_mes_atu, 1, feriados)

    usa_fechado = (
        fechado["ULTIMO_MES_IPCA"] == (data - pd.DateOffset(months=1)).month
        and data < dia_15_mes_atu
        and fechado["INDICE_IPCA_FECHADO_ATUAL"] != fechado["INDICE_IPCA_FECHADO_ANTERIOR"]
    )
    if usa_fechado:
        ipca_usado = (
            (fechado["INDICE_IPCA_FECHADO_ATUAL"] / fechado["INDICE_IPCA_FECHADO_ANTERIOR"]) - 1
        ) * 100
    else:
        ipca_usado = ipca_proj_float

    return {
        "ULTIMO_MES_IPCA": int(fechado["ULTIMO_MES_IPCA"]),
        "INDICE_IPCA_DATA_BASE": INDICE_IPCA_DATA_BASE,
        "INDICE_IPCA_FECHADO_ATUAL": float(fechado["INDICE_IPCA_FECHADO_ATUAL"]),
        "INDICE_IPCA_FECHADO_ANTERIOR": float(fechado["INDICE_IPCA_FECHADO_ANTERIOR"]),
        "VAR_IPCA_ATUAL": float(fechado["VAR_IPCA_ATUAL"]),
        "VAR_IPCA_ANTERIOR": float(fechado["VAR_IPCA_ANTERIOR"]),
        "IPCA_PROJ": float(ipca_proj_float),
        "IPCA_USADO": float(ipca_usado),
    }


def build_ipca_dict(
    as_of: pd.Timestamp | str,
    ipca_monthly: pd.DataFrame,
    projecoes: pd.DataFrame,
    feriados: set[str] | None = None,
) -> dict[str, float | int]:
    feriados = feriados if feriados is not None else _feriados_set()
    inicio, fim = inicio_fim_mes_ipca(as_of, feriados)
    fechado = ipca_fechado_from_monthly(ipca_monthly, as_of)
    proj, proj_ref_month = ipca_proj_float_from_projecoes(projecoes, as_of, fim)
    out = dicionario_ipca(as_of, fechado, proj, feriados)
    out["INICIO_MES_IPCA"] = inicio.strftime("%Y-%m-%d")
    out["FIM_MES_IPCA"] = fim.strftime("%Y-%m-%d")
    out["PROJ_REF_MONTH_USADO"] = proj_ref_month
    return out

In [78]:
data = "2026-05-11"
df_proj = silver_projecoes_e_ipca_ultimos_dois(data)[0]
df_indice = silver_projecoes_e_ipca_ultimos_dois(data)[1]
proj = projecao_mais_recente_valor(df_proj, data)[0]
data_coleta = projecao_mais_recente_valor(df_proj, data)[1]
feriados = _feriados_set()
inicio_fim_mes_ipca(data)
data_coleta


Timestamp('2026-04-28 00:00:00')